# Chapter 13: Neural Network Methods for Partial Differential Equations

## Introduction to Neural Networks

### Implementing a Simple Feedforward Network

In [ ]:
import numpy as np

class SimpleNet:
    """Simple feedforward network with one hidden layer (NumPy)."""

    def __init__(self, input_dim, hidden_dim, output_dim):
        # He initialization, appropriate for ReLU layers
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2.0 / hidden_dim)
        self.b2 = np.zeros((1, output_dim))

    def relu(self, z):
        return np.maximum(0, z)

    def relu_derivative(self, z):
        return (z > 0).astype(float)

    def forward(self, X):
        """Forward pass: X of shape (batch_size, input_dim)."""
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self.relu(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.y_pred = self.z2  # For regression, identity output
        return self.y_pred

    def backward(self, X, y, learning_rate=0.01):
        """Backward pass and parameter update."""
        batch_size = X.shape[0]

        # Output layer gradient
        dz2 = self.y_pred - y
        dW2 = self.a1.T @ dz2 / batch_size
        db2 = np.sum(dz2, axis=0, keepdims=True) / batch_size

        # Hidden layer gradient
        da1 = dz2 @ self.W2.T
        dz1 = da1 * self.relu_derivative(self.z1)
        dW1 = X.T @ dz1 / batch_size
        db1 = np.sum(dz1, axis=0, keepdims=True) / batch_size

        # Update parameters
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1

    def train(self, X, y, epochs=100, learning_rate=0.01):
        """Train the network."""
        for epoch in range(epochs):
            self.forward(X)
            self.backward(X, y, learning_rate)
            if epoch % 400 == 0:
                loss = np.mean((self.y_pred - y)**2)
                print(f"Epoch {epoch}: MSE = {loss:.6f}")

# Example usage
if __name__ == "__main__":
    # Generate synthetic data: y = x^2 + noise
    np.random.seed(42)
    X = np.linspace(-1, 1, 100).reshape(-1, 1)
    y = X**2 + np.random.normal(0, 0.05, (100, 1))

    net = SimpleNet(input_dim=1, hidden_dim=32, output_dim=1)
    net.train(X, y, epochs=2000, learning_rate=0.05)

    # Evaluate
    y_pred = net.forward(X)
    final_loss = np.mean((y_pred - y)**2)
    print(f"Final MSE: {final_loss:.6f}")

## Training Neural Networks

### Training Loop Implementation in PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Define a simple network
class RegressionNet(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.net(x)

def train_network(model, train_loader, test_loader, epochs=100,
                  lr=0.001, lambda_reg=0.0):
    """Train a neural network with Adam optimizer and L2 regularization."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=lambda_reg)
    criterion = nn.MSELoss()

    train_losses = []
    test_losses = []

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X_batch.size(0)

        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        # Validation phase
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                test_loss += loss.item() * X_batch.size(0)

        test_loss /= len(test_loader.dataset)
        test_losses.append(test_loss)

        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{epochs} - "
                  f"Train Loss: {train_loss:.6f}, Test Loss: {test_loss:.6f}")

    return train_losses, test_losses

# Example usage
if __name__ == "__main__":
    # Generate synthetic data
    torch.manual_seed(42)
    X = torch.linspace(-1, 1, 200).reshape(-1, 1)
    y = X**2 + 0.05 * torch.randn_like(X)

    # Create data loaders
    dataset = TensorDataset(X, y)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = torch.utils.data.random_split(
        dataset, [train_size, test_size]
    )

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32)

    # Train network
    model = RegressionNet(input_dim=1, hidden_dim=64)
    train_losses, test_losses = train_network(
        model, train_loader, test_loader, epochs=100, lr=0.001, lambda_reg=0.0001
    )

## Neural Networks as Function Approximators for PDEs

### Automatic Differentiation

In [ ]:
import torch

x = torch.tensor([0.5], requires_grad=True)
u = model(x)
u_x = torch.autograd.grad(u, x, create_graph=True)[0]

# For second derivatives
u_xx = torch.autograd.grad(u_x, x)[0]

### PyTorch Implementation: Approximating a Function

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Define a neural network
class FunctionApproximator(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.net(x)

import numpy as np

def approximate_sine():
    """Approximate u(x) = sin(pi*x) on [-1, 1]."""
    torch.manual_seed(42)

    # Generate training points
    x_train = torch.linspace(-1, 1, 100).reshape(-1, 1)
    y_train = torch.sin(np.pi * x_train)

    model = FunctionApproximator(hidden_dim=32)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()

    # Training loop
    for epoch in range(500):
        optimizer.zero_grad()
        y_pred = model(x_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch+1}: Loss = {loss.item():.6f}")

    # Evaluate
    x_test = torch.linspace(-1, 1, 200).reshape(-1, 1)
    y_test = torch.sin(np.pi * x_test)
    y_pred = model(x_test).detach()

    # Plot
    plt.figure(figsize=(8, 5))
    plt.plot(x_test.numpy(), y_test.numpy(), 'b-',
             label=r'$\sin(\pi x)$', linewidth=2)
    plt.plot(x_test.numpy(), y_pred.numpy(), 'r--',
             label='NN approximation', linewidth=2)
    plt.scatter(x_train.numpy(), y_train.numpy(), c='g', s=20,
                label='Training points')
    plt.xlabel('x')
    plt.ylabel('u(x)')
    plt.legend()
    plt.title(r'Neural Network Approximation of $\sin(\pi x)$')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('approximation.png', dpi=200)

    print("\nFinal test MSE:", criterion(model(x_test), y_test).item())

if __name__ == "__main__":
    approximate_sine()

## Solving the Heat Equation with Neural Networks

### Full PyTorch Implementation: Heat Equation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

class HeatNet(nn.Module):
    """Neural network for heat equation."""
    def __init__(self, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, t):
        """Forward pass: x and t are (N, 1) tensors."""
        xt = torch.cat([x, t], dim=1)  # (N, 2)
        return self.net(xt)

def solve_heat_equation(alpha=0.1, T_final=1.0, epochs=2000):
    """Solve heat equation u_t = alpha * u_xx with analytical solution."""
    torch.manual_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Analytical solution
    def u_exact(x, t):
        return torch.exp(-alpha * np.pi**2 * t) * torch.sin(np.pi * x)

    # Generate collocation points
    N_c = 1000  # interior collocation points
    N_b = 100   # boundary collocation points
    N_i = 100   # initial condition points

    # Interior points (random sampling)
    x_c = torch.FloatTensor(N_c, 1).uniform_(0.01, 0.99).to(device)
    t_c = torch.FloatTensor(N_c, 1).uniform_(0.01, T_final).to(device)

    # Boundary points (x=0 and x=1)
    x_b1 = torch.zeros(N_b // 2, 1).to(device)
    x_b2 = torch.ones(N_b // 2, 1).to(device)
    t_b = torch.FloatTensor(N_b // 2, 1).uniform_(0, T_final).to(device)
    x_b = torch.cat([x_b1, x_b2], dim=0)
    t_b = torch.cat([t_b, t_b], dim=0)

    # Initial condition points (t=0)
    x_i = torch.FloatTensor(N_i, 1).uniform_(0, 1).to(device)
    t_i = torch.zeros(N_i, 1).to(device)
    u_i = u_exact(x_i, t_i).to(device)

    # Initialize network
    model = HeatNet(hidden_dim=128).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Training loop
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        # Requires grad for derivatives
        x_c.requires_grad_(True)
        t_c.requires_grad_(True)

        u_c = model(x_c, t_c)

        # Compute derivatives
        u_t = torch.autograd.grad(u_c, t_c, grad_outputs=torch.ones_like(u_c),
                                   create_graph=True)[0]
        u_x = torch.autograd.grad(u_c, x_c, grad_outputs=torch.ones_like(u_c),
                                   create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x, x_c, grad_outputs=torch.ones_like(u_x),
                                    create_graph=True)[0]

        # PDE residual
        pde_residual = u_t - alpha * u_xx
        loss_pde = torch.mean(pde_residual**2)

        # Boundary condition loss
        u_b = model(x_b, t_b)
        loss_bc = torch.mean(u_b**2)  # u = 0 at x = 0, 1

        # Initial condition loss
        u_init = model(x_i, t_i)
        loss_ic = torch.mean((u_init - u_i)**2)

        # Total loss
        loss = loss_pde + 10 * loss_bc + 10 * loss_ic

        loss.backward()
        optimizer.step()

        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}/{epochs}: "
                  f"Loss_PDE={loss_pde.item():.6f}, "
                  f"Loss_BC={loss_bc.item():.6f}, "
                  f"Loss_IC={loss_ic.item():.6f}")

        # Detach for next iteration
        x_c = x_c.detach()
        t_c = t_c.detach()

    # Evaluate on a grid
    model.eval()
    with torch.no_grad():
        x_grid = torch.linspace(0, 1, 100).reshape(-1, 1).to(device)
        t_grid = torch.linspace(0, T_final, 50).reshape(-1, 1).to(device)

        u_pred = np.zeros((len(x_grid), len(t_grid)))
        u_true = np.zeros((len(x_grid), len(t_grid)))

        for j, t_val in enumerate(t_grid):
            x_test = x_grid
            t_test = t_val.repeat(len(x_grid), 1)
            u_pred[:, j] = model(x_test, t_test).cpu().numpy().ravel()
            u_true[:, j] = u_exact(x_test, t_test).numpy().ravel()

    # Plot comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    X_plot, T_plot = np.meshgrid(x_grid.cpu().numpy().ravel(),
                                   t_grid.cpu().numpy().ravel(), indexing='ij')

    im1 = axes[0].contourf(T_plot, X_plot, u_pred, levels=20, cmap='viridis')
    axes[0].set_title('Neural Network Solution')
    axes[0].set_xlabel('t')
    axes[0].set_ylabel('x')
    fig.colorbar(im1, ax=axes[0])

    im2 = axes[1].contourf(T_plot, X_plot, u_true, levels=20, cmap='viridis')
    axes[1].set_title('Analytical Solution')
    axes[1].set_xlabel('t')
    axes[1].set_ylabel('x')
    fig.colorbar(im2, ax=axes[1])

    plt.tight_layout()
    plt.savefig('heat_equation_solution.png', dpi=100)

    # Compute L2 error
    l2_error = np.sqrt(np.mean((u_pred - u_true)**2))
    print(f"\nL2 error: {l2_error:.6e}")

if __name__ == "__main__":
    solve_heat_equation(alpha=0.1, T_final=1.0, epochs=2000)

## Solving the Wave Equation with Neural Networks

### PyTorch Implementation: Wave Equation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

class WaveNet(nn.Module):
    """Neural network for wave equation."""
    def __init__(self, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, t):
        xt = torch.cat([x, t], dim=1)
        return self.net(xt)

def solve_wave_equation(c=1.0, T_final=1.0, epochs=3000):
    """Solve wave equation u_tt = c^2 * u_xx."""
    torch.manual_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Analytical solution
    def u_exact(x, t):
        return torch.cos(np.pi * c * t) * torch.sin(np.pi * x)

    # Generate collocation points
    N_c = 1500
    N_b = 150
    N_i = 150

    # Interior
    x_c = torch.FloatTensor(N_c, 1).uniform_(0.01, 0.99).to(device)
    t_c = torch.FloatTensor(N_c, 1).uniform_(0.01, T_final).to(device)

    # Boundary
    x_b = torch.cat([torch.zeros(N_b//2, 1), torch.ones(N_b//2, 1)]).to(device)
    t_b = torch.FloatTensor(N_b, 1).uniform_(0, T_final).to(device)

    # Initial displacement and velocity
    x_i = torch.FloatTensor(N_i, 1).uniform_(0, 1).to(device)
    t_i = torch.zeros(N_i, 1).to(device)
    u_i = u_exact(x_i, t_i).to(device)
    v_i = torch.zeros(N_i, 1).to(device)  # Initial velocity is zero

    model = WaveNet(hidden_dim=128).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        x_c.requires_grad_(True)
        t_c.requires_grad_(True)

        u_c = model(x_c, t_c)

        # First derivatives
        u_t = torch.autograd.grad(u_c, t_c, grad_outputs=torch.ones_like(u_c),
                                   create_graph=True)[0]
        u_x = torch.autograd.grad(u_c, x_c, grad_outputs=torch.ones_like(u_c),
                                   create_graph=True)[0]

        # Second derivatives
        u_tt = torch.autograd.grad(u_t, t_c, grad_outputs=torch.ones_like(u_t),
                                    create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x, x_c, grad_outputs=torch.ones_like(u_x),
                                    create_graph=True)[0]

        # PDE residual
        pde_residual = u_tt - c**2 * u_xx
        loss_pde = torch.mean(pde_residual**2)

        # Boundary loss
        u_b = model(x_b, t_b)
        loss_bc = torch.mean(u_b**2)

        # Initial displacement loss
        t_i.requires_grad_(True)
        u_0 = model(x_i, t_i)
        loss_u0 = torch.mean((u_0 - u_i)**2)

        # Initial velocity loss: u_t(x, 0) = 0
        u_t_0 = torch.autograd.grad(u_0, t_i,
                                     grad_outputs=torch.ones_like(u_0),
                                     create_graph=True)[0]
        loss_v0 = torch.mean((u_t_0 - v_i)**2)

        loss = loss_pde + 10 * loss_bc + 10 * loss_u0 + 10 * loss_v0

        loss.backward()
        optimizer.step()

        if (epoch + 1) % 300 == 0:
            print(f"Epoch {epoch+1}/{epochs}: "
                  f"Loss_PDE={loss_pde.item():.6f}, "
                  f"Loss_BC={loss_bc.item():.6f}")

        x_c = x_c.detach()
        t_c = t_c.detach()

    # Evaluate
    model.eval()
    with torch.no_grad():
        x_grid = torch.linspace(0, 1, 100).reshape(-1, 1).to(device)
        t_grid = torch.linspace(0, T_final, 50).reshape(-1, 1).to(device)

        u_pred = np.zeros((len(x_grid), len(t_grid)))
        u_true = np.zeros((len(x_grid), len(t_grid)))

        for j, t_val in enumerate(t_grid):
            x_test = x_grid
            t_test = t_val.repeat(len(x_grid), 1)
            u_pred[:, j] = model(x_test, t_test).cpu().numpy().ravel()
            u_true[:, j] = u_exact(x_test, t_test).numpy().ravel()

    l2_error = np.sqrt(np.mean((u_pred - u_true)**2))
    print(f"\nL2 error: {l2_error:.6e}")

if __name__ == "__main__":
    solve_wave_equation(c=1.0, T_final=1.0, epochs=3000)

## Solving Elliptic PDEs with Neural Networks

### PyTorch Implementation: 2D Poisson Equation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

class PoissonNet2D(nn.Module):
    """Neural network for 2D Poisson equation."""
    def __init__(self, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, y):
        xy = torch.cat([x, y], dim=1)
        return self.net(xy)

def solve_poisson_2d(epochs=5000):
    """Solve -u_xx - u_yy = 8*pi^2*sin(2*pi*x)*sin(2*pi*y) on [0,1]^2."""
    torch.manual_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Analytical solution
    def u_exact(x, y):
        return torch.sin(2 * np.pi * x) * torch.sin(2 * np.pi * y)

    def f_rhs(x, y):
        return 8 * np.pi**2 * torch.sin(2 * np.pi * x) * torch.sin(2 * np.pi * y)

    # Generate collocation points
    N_c = 2000  # Interior
    N_b = 400   # Boundary

    # Interior points
    x_c = torch.FloatTensor(N_c, 1).uniform_(0.01, 0.99).to(device)
    y_c = torch.FloatTensor(N_c, 1).uniform_(0.01, 0.99).to(device)
    f_c = f_rhs(x_c, y_c).to(device)

    # Boundary points (four edges)
    x_b_list = []
    y_b_list = []

    # Bottom and top edges (y=0 or y=1, x varies)
    x_edge = torch.FloatTensor(N_b//4, 1).uniform_(0, 1)
    x_b_list.extend([x_edge, x_edge])
    y_b_list.extend([torch.zeros(N_b//4, 1), torch.ones(N_b//4, 1)])

    # Left and right edges (x=0 or x=1, y varies)
    y_edge = torch.FloatTensor(N_b//4, 1).uniform_(0, 1)
    x_b_list.extend([torch.zeros(N_b//4, 1), torch.ones(N_b//4, 1)])
    y_b_list.extend([y_edge, y_edge])

    x_b = torch.cat(x_b_list).to(device)
    y_b = torch.cat(y_b_list).to(device)
    u_b = u_exact(x_b, y_b).to(device)

    model = PoissonNet2D(hidden_dim=128).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        x_c.requires_grad_(True)
        y_c.requires_grad_(True)

        u_c = model(x_c, y_c)

        # Compute Laplacian
        u_x = torch.autograd.grad(u_c, x_c, grad_outputs=torch.ones_like(u_c),
                                   create_graph=True)[0]
        u_y = torch.autograd.grad(u_c, y_c, grad_outputs=torch.ones_like(u_c),
                                   create_graph=True)[0]

        u_xx = torch.autograd.grad(u_x, x_c, grad_outputs=torch.ones_like(u_x),
                                    create_graph=True)[0]
        u_yy = torch.autograd.grad(u_y, y_c, grad_outputs=torch.ones_like(u_y),
                                    create_graph=True)[0]

        # PDE residual: -u_xx - u_yy - f = 0
        pde_residual = -u_xx - u_yy - f_c
        loss_pde = torch.mean(pde_residual**2)

        # Boundary loss
        u_pred_b = model(x_b, y_b)
        loss_bc = torch.mean((u_pred_b - u_b)**2)

        loss = loss_pde + 10 * loss_bc

        loss.backward()
        optimizer.step()

        if (epoch + 1) % 500 == 0:
            print(f"Epoch {epoch+1}/{epochs}: "
                  f"Loss_PDE={loss_pde.item():.6f}, "
                  f"Loss_BC={loss_bc.item():.6f}")

        x_c = x_c.detach()
        y_c = y_c.detach()

    # Evaluate on grid
    model.eval()
    with torch.no_grad():
        grid_size = 50
        x_grid = np.linspace(0, 1, grid_size)
        y_grid = np.linspace(0, 1, grid_size)
        X_grid, Y_grid = np.meshgrid(x_grid, y_grid)

        x_flat = torch.FloatTensor(X_grid.ravel()).reshape(-1, 1).to(device)
        y_flat = torch.FloatTensor(Y_grid.ravel()).reshape(-1, 1).to(device)

        u_pred = model(x_flat, y_flat).cpu().numpy().reshape(X_grid.shape)
        u_true = u_exact(x_flat, y_flat).cpu().numpy().reshape(X_grid.shape)

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    im1 = axes[0].contourf(X_grid, Y_grid, u_pred, levels=20, cmap='viridis')
    axes[0].set_title('NN Solution')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('y')
    fig.colorbar(im1, ax=axes[0])

    im2 = axes[1].contourf(X_grid, Y_grid, u_true, levels=20, cmap='viridis')
    axes[1].set_title('Exact Solution')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('y')
    fig.colorbar(im2, ax=axes[1])

    error = np.abs(u_pred - u_true)
    im3 = axes[2].contourf(X_grid, Y_grid, error, levels=20, cmap='hot')
    axes[2].set_title('Absolute Error')
    axes[2].set_xlabel('x')
    axes[2].set_ylabel('y')
    fig.colorbar(im3, ax=axes[2])

    plt.tight_layout()
    plt.savefig('poisson_2d_solution.png', dpi=100)

    l2_error = np.sqrt(np.mean((u_pred - u_true)**2))
    print(f"\nL2 error: {l2_error:.6e}")

if __name__ == "__main__":
    solve_poisson_2d(epochs=5000)

## Transfer Learning for PDEs

### PyTorch Implementation: Transfer Learning for Heat Equation Variants

In [ ]:
import time

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

class HeatNetTransferable(nn.Module):
    """Heat equation network designed for transfer learning."""
    def __init__(self, hidden_dim=128):
        super().__init__()
        # Feature extraction layers (early layers)
        self.feature_layers = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()
        )
        # Task-specific output layer
        self.output_layer = nn.Linear(hidden_dim, 1)

    def forward(self, x, t):
        xt = torch.cat([x, t], dim=1)
        features = self.feature_layers(xt)
        return self.output_layer(features)

    def pde_residual(self, x, t, alpha):
        """Compute heat equation residual."""
        x.requires_grad_(True)
        t.requires_grad_(True)

        u = self.forward(x, t)
        u_t = torch.autograd.grad(u.sum(), t, create_graph=True)[0]
        u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]

        return u_t - alpha * u_xx

def train_transfer_learning(source_alpha=0.1, target_alpha=0.05,
                           source_epochs=2000, target_epochs=500):
    """Train on source task, then transfer to target task."""
    torch.manual_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Data generation
    def generate_data(alpha, n_residual=1000, n_boundary=100, n_initial=100):
        x_residual = torch.rand(n_residual, 1).to(device)
        t_residual = torch.rand(n_residual, 1).to(device)

        x_boundary = torch.cat([torch.zeros(n_boundary//2, 1),
                               torch.ones(n_boundary//2, 1)]).to(device)
        t_boundary = torch.rand(n_boundary, 1).to(device)

        x_initial = torch.rand(n_initial, 1).to(device)
        t_initial = torch.zeros(n_initial, 1).to(device)
        u_initial = torch.sin(np.pi * x_initial).to(device)

        return (x_residual, t_residual), (x_boundary, t_boundary), \
               (x_initial, t_initial, u_initial)

    # Phase 1: Train on source task
    print(f"\n=== Source Task: Heat equation with alpha={source_alpha} ===")
    model_source = HeatNetTransferable().to(device)
    optimizer_source = optim.Adam(model_source.parameters(), lr=0.001)

    (x_r_src, t_r_src), (x_b_src, t_b_src), \
    (x_i_src, t_i_src, u_i_src) = generate_data(source_alpha)

    t_start = time.time()

    for epoch in range(source_epochs):
        optimizer_source.zero_grad()

        residual = model_source.pde_residual(x_r_src, t_r_src, source_alpha)
        loss_pde = torch.mean(residual ** 2)

        u_b = model_source(x_b_src, t_b_src)
        loss_bc = torch.mean(u_b ** 2)

        u_i = model_source(x_i_src, t_i_src)
        loss_ic = torch.mean((u_i - u_i_src) ** 2)

        loss = loss_pde + 10 * loss_bc + 10 * loss_ic
        loss.backward()
        optimizer_source.step()

        if (epoch + 1) % 400 == 0:
            print(f"Source Epoch {epoch+1}/{source_epochs}: Loss={loss.item():.6e}")

    source_time = time.time() - t_start

    # Phase 2: Transfer learning - fine-tuning
    print(f"\n=== Target Task (Fine-tuning): Heat equation with alpha={target_alpha} ===")
    model_target_ft = HeatNetTransferable().to(device)
    # Copy weights from source
    model_target_ft.load_state_dict(model_source.state_dict())

    optimizer_target_ft = optim.Adam(model_target_ft.parameters(), lr=0.0001)  # Lower LR

    (x_r_tgt, t_r_tgt), (x_b_tgt, t_b_tgt), \
    (x_i_tgt, t_i_tgt, u_i_tgt) = generate_data(target_alpha)

    t_start = time.time()

    for epoch in range(target_epochs):
        optimizer_target_ft.zero_grad()

        residual = model_target_ft.pde_residual(x_r_tgt, t_r_tgt, target_alpha)
        loss_pde = torch.mean(residual ** 2)

        u_b = model_target_ft(x_b_tgt, t_b_tgt)
        loss_bc = torch.mean(u_b ** 2)

        u_i = model_target_ft(x_i_tgt, t_i_tgt)
        loss_ic = torch.mean((u_i - u_i_tgt) ** 2)

        loss = loss_pde + 10 * loss_bc + 10 * loss_ic
        loss.backward()
        optimizer_target_ft.step()

        if (epoch + 1) % 100 == 0:
            print(f"Target Epoch {epoch+1}/{target_epochs}: Loss={loss.item():.6e}")

    transfer_time = time.time() - t_start

    # Phase 3: Train from scratch for comparison
    print(f"\n=== Control: Training from scratch with alpha={target_alpha} ===")
    model_target_scratch = HeatNetTransferable().to(device)
    optimizer_target_scratch = optim.Adam(model_target_scratch.parameters(), lr=0.001)

    t_start = time.time()

    for epoch in range(target_epochs):
        optimizer_target_scratch.zero_grad()

        residual = model_target_scratch.pde_residual(x_r_tgt, t_r_tgt, target_alpha)
        loss_pde = torch.mean(residual ** 2)

        u_b = model_target_scratch(x_b_tgt, t_b_tgt)
        loss_bc = torch.mean(u_b ** 2)

        u_i = model_target_scratch(x_i_tgt, t_i_tgt)
        loss_ic = torch.mean((u_i - u_i_tgt) ** 2)

        loss = loss_pde + 10 * loss_bc + 10 * loss_ic
        loss.backward()
        optimizer_target_scratch.step()

        if (epoch + 1) % 100 == 0:
            print(f"Scratch Epoch {epoch+1}/{target_epochs}: Loss={loss.item():.6e}")

    scratch_time = time.time() - t_start

    print("\n=== Timing Summary ===")
    print(f"Source training time: {source_time:.2f}s")
    print(f"Transfer learning time: {transfer_time:.2f}s")
    print(f"Training from scratch time: {scratch_time:.2f}s")
    print(f"Speedup from transfer: {scratch_time / transfer_time:.2f}x")

if __name__ == "__main__":
    train_transfer_learning(source_alpha=0.1, target_alpha=0.05)

## Training Strategies and Hyperparameter Tuning

### Residual-Based Adaptive Sampling

In [ ]:
def adaptive_sampling_loop(model, x_domain, t_domain, n_initial=1000,
                          n_iterations=5, refinement_fraction=0.2):
    """Residual-based adaptive refinement sampling."""
    x_residual = torch.rand(n_initial, 1) * (x_domain[1] - x_domain[0]) + x_domain[0]
    t_residual = torch.rand(n_initial, 1) * (t_domain[1] - t_domain[0]) + t_domain[0]

    for iteration in range(n_iterations):
        print(f"Adaptive iteration {iteration+1}/{n_iterations}")

        # Train for several epochs with current samples
        for epoch in range(100):
            # ... training code ...
            pass

        # Compute residuals at all points
        with torch.no_grad():
            residuals = model.pde_residual(x_residual, t_residual)
            residual_magnitudes = torch.abs(residuals.detach())

        # Normalize and create probability distribution
        residual_prob = residual_magnitudes / residual_magnitudes.sum()

        # Sample new points preferentially from high-residual regions
        n_new = int(refinement_fraction * n_initial)
        indices = torch.multinomial(residual_prob.squeeze(), n_new, replacement=True)

        # Perturb around high-residual points
        x_new = x_residual[indices] + 0.01 * torch.randn_like(x_residual[indices])
        t_new = t_residual[indices] + 0.01 * torch.randn_like(t_residual[indices])

        # Keep some old points for stability
        keep_frac = 0.8
        keep_indices = torch.randperm(len(x_residual))[:int(keep_frac * len(x_residual))]
        x_residual = torch.cat([x_residual[keep_indices], x_new])
        t_residual = torch.cat([t_residual[keep_indices], t_new])

    return x_residual, t_residual

## Residual Networks and Skip Connections for PDEs

### PyTorch Implementation of Deep PDE Solver with ResNet

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

class ResidualBlock(nn.Module):
    """Residual block: y = F(x) + x"""

    def __init__(self, hidden_dim, activation=nn.Tanh):
        super().__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.act = activation()

    def forward(self, x):
        residual = x
        out = self.fc1(x)
        out = self.act(out)
        out = self.fc2(out)
        out = out + residual  # Skip connection
        return out

class ResidualNetworkPDE(nn.Module):
    """Deep neural network with residual blocks for solving PDEs."""

    def __init__(self, in_dim, out_dim, hidden_dim=64, n_blocks=8):
        super().__init__()
        self.fc_in = nn.Linear(in_dim, hidden_dim)
        self.blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim) for _ in range(n_blocks)]
        )
        self.fc_out = nn.Linear(hidden_dim, out_dim)
        self.act = nn.Tanh()

    def forward(self, x):
        x = self.fc_in(x)
        x = self.act(x)
        x = self.blocks(x)
        x = self.fc_out(x)
        return x

class HighwayBlock(nn.Module):
    """Highway network block with learned gating."""

    def __init__(self, hidden_dim):
        super().__init__()
        self.fc_transform = nn.Linear(hidden_dim, hidden_dim)
        self.fc_gate = nn.Linear(hidden_dim, hidden_dim)
        self.act = nn.Tanh()

    def forward(self, x):
        transform = self.act(self.fc_transform(x))
        gate = torch.sigmoid(self.fc_gate(x))
        return gate * transform + (1 - gate) * x

class HighwayNetworkPDE(nn.Module):
    """PDE solver using highway networks."""

    def __init__(self, in_dim, out_dim, hidden_dim=64, n_blocks=8):
        super().__init__()
        self.fc_in = nn.Linear(in_dim, hidden_dim)
        self.blocks = nn.Sequential(
            *[HighwayBlock(hidden_dim) for _ in range(n_blocks)]
        )
        self.fc_out = nn.Linear(hidden_dim, out_dim)

    def forward(self, x):
        x = self.fc_in(x)
        x = self.blocks(x)
        x = self.fc_out(x)
        return x

# Test on 1D heat equation: u_t = u_xx
# Generate synthetic data
torch.manual_seed(42)
np.random.seed(42)

n_train = 5000
x_train = torch.rand(n_train, 1) - 0.5  # x uniform in [-0.5, 0.5]
t_train = torch.rand(n_train, 1)        # t uniform in [0, 1]

# Analytical solution: u(x,t) = exp(-pi^2*t) * sin(pi*x)
def analytical_solution(x, t):
    return torch.exp(-np.pi**2 * t) * torch.sin(np.pi * x)

u_true = analytical_solution(x_train, t_train)

# Create models
model_resnet = ResidualNetworkPDE(2, 1, hidden_dim=64, n_blocks=8)
model_highway = HighwayNetworkPDE(2, 1, hidden_dim=64, n_blocks=8)

# Training
def train_model(model, x, t, u_true, epochs=500, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []

    for epoch in range(epochs):
        inputs = torch.cat([x, t], dim=1)
        outputs = model(inputs)

        loss = nn.MSELoss()(outputs, u_true)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch+1}, Loss: {loss.item():.6e}")

    return losses

print("Training ResNet-based PDE solver...")
losses_resnet = train_model(model_resnet, x_train, t_train, u_true)

print("\nTraining Highway Network-based PDE solver...")
losses_highway = train_model(model_highway, x_train, t_train, u_true)

# Compare on test set
x_test = torch.linspace(-0.5, 0.5, 100).reshape(-1, 1)
t_test = torch.linspace(0, 1, 50).reshape(-1, 1)

X_test, T_test = torch.meshgrid(x_test.squeeze(), t_test.squeeze(),
                                indexing='ij')
inputs_test = torch.stack([X_test.flatten(), T_test.flatten()], dim=1)

with torch.no_grad():
    u_resnet = model_resnet(inputs_test).reshape(X_test.shape)
    u_highway = model_highway(inputs_test).reshape(X_test.shape)
    u_analytical = analytical_solution(X_test, T_test)

error_resnet = torch.mean((u_resnet - u_analytical)**2).item()
error_highway = torch.mean((u_highway - u_analytical)**2).item()

print(f"\nTest MSE - ResNet: {error_resnet:.6e}")
print(f"Test MSE - Highway: {error_highway:.6e}")

# Plot
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Loss curves
axes[0, 0].semilogy(losses_resnet, 'b-', label='ResNet', linewidth=2)
axes[0, 0].semilogy(losses_highway, 'r-', label='Highway', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=11)
axes[0, 0].set_ylabel('Training Loss', fontsize=11)
axes[0, 0].set_title('Convergence Comparison', fontsize=12)
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Analytical solution
im1 = axes[0, 1].contourf(X_test.numpy(), T_test.numpy(),
                          u_analytical.numpy())
axes[0, 1].set_title('Analytical Solution', fontsize=12)
axes[0, 1].set_xlabel('x', fontsize=11)
axes[0, 1].set_ylabel('t', fontsize=11)
plt.colorbar(im1, ax=axes[0, 1])

# ResNet prediction
im2 = axes[1, 0].contourf(X_test.numpy(), T_test.numpy(),
                          u_resnet.numpy())
axes[1, 0].set_title(f'ResNet Prediction (MSE={error_resnet:.2e})',
                     fontsize=12)
axes[1, 0].set_xlabel('x', fontsize=11)
axes[1, 0].set_ylabel('t', fontsize=11)
plt.colorbar(im2, ax=axes[1, 0])

# Highway prediction
im3 = axes[1, 1].contourf(X_test.numpy(), T_test.numpy(),
                          u_highway.numpy())
axes[1, 1].set_title(f'Highway Prediction (MSE={error_highway:.2e})',
                     fontsize=12)
axes[1, 1].set_xlabel('x', fontsize=11)
axes[1, 1].set_ylabel('t', fontsize=11)
plt.colorbar(im3, ax=axes[1, 1])

plt.tight_layout()
plt.show()

## Multi-Scale Neural Networks

### Multi-Resolution Architecture

In [ ]:
class FourierFeatureEmbedding(nn.Module):
    """Fourier feature embedding for PDE inputs."""

    def __init__(self, in_dim, embed_dim, scale=1.0):
        super().__init__()
        # Random Fourier features
        self.B = nn.Parameter(torch.randn(in_dim, embed_dim) * scale,
                             requires_grad=False)
        self.embed_dim = embed_dim

    def forward(self, x):
        """Apply Fourier embedding: x -> [sin(2pi*B*x), cos(2pi*B*x)]"""
        proj = torch.matmul(x, self.B)
        return torch.cat([torch.sin(2 * np.pi * proj),
                         torch.cos(2 * np.pi * proj)], dim=1)

class MultiScaleNetwork(nn.Module):
    """Multi-scale neural network with Fourier features."""

    def __init__(self, in_dim, out_dim, hidden_dim=64,
                embed_dim_list=[16, 32, 64]):
        super().__init__()

        # Multiple embedding scales
        self.embeddings = nn.ModuleList([
            FourierFeatureEmbedding(in_dim, embed_dim, scale=s)
            for s, embed_dim in zip([1.0, 10.0, 100.0], embed_dim_list)
        ])

        total_embed = sum(embed_dim_list) * 2  # sin and cos

        # Separate pathways for each scale
        self.pathways = nn.ModuleList([
            nn.Sequential(
                nn.Linear(2*embed_dim, hidden_dim),
                nn.Tanh(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.Tanh(),
                nn.Linear(hidden_dim, out_dim)
            )
            for embed_dim in embed_dim_list
        ])

        # Fusion layer
        self.fusion = nn.Sequential(
            nn.Linear(out_dim * len(embed_dim_list), hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x):
        # Process each scale independently
        outputs = []
        for embedding, pathway in zip(self.embeddings, self.pathways):
            embedded = embedding(x)
            out = pathway(embedded)
            outputs.append(out)

        # Fuse multi-scale outputs
        fused_input = torch.cat(outputs, dim=1)
        final_output = self.fusion(fused_input)

        return final_output

# Example: Train multi-scale network on advection-diffusion
# u_t + u_x = 0.01 * u_xx

model_multiscale = MultiScaleNetwork(2, 1, hidden_dim=128,
                                    embed_dim_list=[16, 32, 64])
optimizer = optim.Adam(model_multiscale.parameters(), lr=0.001)

# Synthetic data (same as before)
losses_multiscale = []
for epoch in range(500):
    inputs = torch.cat([x_train, t_train], dim=1)
    outputs = model_multiscale(inputs)

    loss = nn.MSELoss()(outputs, u_true)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses_multiscale.append(loss.item())

    if (epoch + 1) % 100 == 0:
        print(f"MultiScale - Epoch {epoch+1}, Loss: {loss.item():.6e}")

# Test
with torch.no_grad():
    u_multiscale = model_multiscale(inputs_test).reshape(X_test.shape)
    error_multiscale = torch.mean((u_multiscale - u_analytical)**2).item()

print(f"\nTest MSE - MultiScale: {error_multiscale:.6e}")

## Neural Operator Methods

### Training FNO on Parametric PDEs

In [ ]:
class FourierNeuralOperatorLayer(nn.Module):
    """Single FNO layer: apply linear operator in Fourier space."""

    def __init__(self, in_channels, out_channels, modes=16):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes = modes

        # Learnable weights in Fourier space
        self.weight = nn.Parameter(
            torch.randn(in_channels, out_channels, modes, dtype=torch.cfloat)
        )

    def forward(self, x):
        """x shape: (batch, in_channels, spatial_dim)"""
        # FFT
        x_ft = torch.fft.rfft(x, dim=-1)

        # Apply operator in Fourier space
        # Keep only the first self.modes frequencies
        x_ft_truncated = x_ft[..., :self.modes]

        # Multiply by learnable weights (complex multiplication)
        out_ft = torch.einsum('bci,coi->boi', x_ft_truncated, self.weight)

        # Pad back to full size
        out_ft_full = torch.zeros(x.shape[0], self.out_channels,
                                 x.shape[-1]//2 + 1, dtype=torch.cfloat,
                                 device=x.device)
        out_ft_full[..., :self.modes] = out_ft

        # Inverse FFT
        out = torch.fft.irfft(out_ft_full, n=x.shape[-1])

        return out

class FourierNeuralOperator(nn.Module):
    """Full FNO network for solving parametric PDEs."""

    def __init__(self, in_channels, out_channels, modes=16, layers=4,
                hidden_dim=64):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels

        # Input projection
        self.fc0 = nn.Linear(in_channels, hidden_dim)

        # FNO layers
        self.fno_layers = nn.ModuleList([
            FourierNeuralOperatorLayer(hidden_dim, hidden_dim, modes=modes)
            for _ in range(layers)
        ])

        # Local linear layers (skip connection projections)
        self.local_layers = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim) for _ in range(layers)
        ])

        # Output projection
        self.fc1 = nn.Linear(hidden_dim, out_channels)

    def forward(self, x):
        """x shape: (batch, spatial_dim, in_channels)"""
        x = x.transpose(1, 2)  # (batch, in_channels, spatial_dim)
        x = self.fc0(x.transpose(1, 2)).transpose(1, 2)
        # Now x: (batch, hidden_dim, spatial_dim)

        for fno_layer, local_layer in zip(self.fno_layers, self.local_layers):
            # FNO: operates in Fourier space
            x_fno = fno_layer(x)

            # Local: applies pointwise to each spatial location
            x_local = local_layer(x.transpose(1, 2)).transpose(1, 2)

            # Combine
            x = x_fno + x_local

        # Output projection
        x = x.transpose(1, 2)  # (batch, spatial_dim, hidden_dim)
        x = self.fc1(x)  # (batch, spatial_dim, out_channels)

        return x

# Example: Train FNO for Burgers' equation with varying viscosity
# u_t + u*u_x = nu*u_xx

model_fno = FourierNeuralOperator(in_channels=1, out_channels=1,
                                 modes=16, layers=4, hidden_dim=64)

# Generate synthetic training data for Burgers' equation
def generate_burgers_data(n_samples=100, nu=0.01):
    """Generate Burgers' solutions with a stable explicit FD scheme."""
    nx, nt = 256, 1000
    x = np.linspace(-1, 1, nx)
    dx = x[1] - x[0]
    dt = 1.0 / nt  # satisfies dt < dx^2/(2*nu) and the advective CFL

    # Initial conditions: Gaussian pulses at random positions
    a_vals = np.random.uniform(0.5, 2, n_samples)
    x0_vals = np.random.uniform(-0.5, 0.5, n_samples)

    u_init, u_final = [], []
    for a, x0 in zip(a_vals, x0_vals):
        u = a * np.exp(-20*(x - x0)**2)
        u_init.append(u.copy())

        # March to t = 1 (upwind advection + central diffusion)
        for _ in range(nt):
            u_x = (u[1:-1] - u[:-2]) / dx
            u_xx = (u[2:] - 2*u[1:-1] + u[:-2]) / dx**2
            u[1:-1] += dt * (-u[1:-1] * u_x + nu * u_xx)
            u[0], u[-1] = u[1], u[-2]  # zero-gradient ends

        u_final.append(u.copy())

    return np.array(u_init), np.array(u_final)

u_init, u_final = generate_burgers_data(n_samples=100)

# Convert to tensors
u_init_tensor = torch.from_numpy(u_init).float().unsqueeze(-1)
u_final_tensor = torch.from_numpy(u_final).float().unsqueeze(-1)

# Training
optimizer_fno = optim.Adam(model_fno.parameters(), lr=0.001)
losses_fno = []

for epoch in range(200):
    optimizer_fno.zero_grad()

    u_pred = model_fno(u_init_tensor)
    loss = nn.MSELoss()(u_pred, u_final_tensor)

    loss.backward()
    optimizer_fno.step()

    losses_fno.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(f"FNO - Epoch {epoch+1}, Loss: {loss.item():.6e}")

print("\nFNO training complete.")

## Comprehensive Worked Examples

### Training Procedure

In [ ]:
# Complete Allen-Cahn solver using neural networks
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import grad
import numpy as np
import matplotlib.pyplot as plt

class AllenCahnSolver(nn.Module):
    """Neural network solver for Allen-Cahn equation."""

    def __init__(self, hidden_dim=64, n_layers=4):
        super().__init__()
        layers = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        xt = torch.cat([x, t], dim=1)
        return self.net(xt)

    def compute_residual(self, x, t):
        """Compute PDE residual."""
        u = self(x, t)

        # Compute derivatives using autograd
        u_t = grad(u.sum(), t, create_graph=True)[0]
        u_x = grad(u.sum(), x, create_graph=True)[0]
        u_xx = grad(u_x.sum(), x, create_graph=True)[0]

        # PDE: u_t = 0.0001*u_xx + 5*u*(1-u^2)
        residual = u_t - 0.0001 * u_xx - 5*u*(1 - u**2)

        return residual

def solve_allen_cahn():
    """Solve Allen-Cahn equation using neural network."""
    torch.manual_seed(42)

    # Generate training data
    n_residual = 10000  # Residual points
    n_boundary = 1000   # Boundary points
    n_initial = 1000    # Initial condition points

    # Residual collocation points
    x_r = (torch.rand(n_residual, 1) * 2 - 1).requires_grad_(True)
    t_r = torch.rand(n_residual, 1).requires_grad_(True)

    # Initial condition: u(x, 0) = x^2 * cos(pi*x)
    x_i = (torch.rand(n_initial, 1) * 2 - 1)
    t_i = torch.zeros(n_initial, 1)
    u_i = x_i**2 * torch.cos(np.pi * x_i)

    # Boundary conditions: u(-1, t) = -1, u(1, t) = 1
    t_b = torch.rand(n_boundary, 1)
    u_b_left = -torch.ones(n_boundary//2, 1)
    u_b_right = torch.ones(n_boundary//2, 1)
    x_b_left = -torch.ones(n_boundary//2, 1)
    x_b_right = torch.ones(n_boundary//2, 1)
    t_b_left = t_b[:n_boundary//2]
    t_b_right = t_b[n_boundary//2:]

    # Create model and optimizer
    model = AllenCahnSolver(hidden_dim=64, n_layers=4)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    losses = []

    for epoch in range(1000):
        optimizer.zero_grad()

        # Residual loss
        residual = model.compute_residual(x_r, t_r)
        loss_residual = torch.mean(residual**2)

        # Initial condition loss
        u_pred_i = model(x_i, t_i)
        loss_initial = torch.mean((u_pred_i - u_i)**2)

        # Boundary condition loss
        u_pred_bl = model(x_b_left, t_b_left)
        u_pred_br = model(x_b_right, t_b_right)
        loss_boundary = (torch.mean((u_pred_bl - u_b_left)**2) +
                        torch.mean((u_pred_br - u_b_right)**2))

        # Total loss
        loss = loss_residual + loss_initial + loss_boundary

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch+1}, Loss: {loss.item():.6e}")

    return model, losses

model_ac, losses_ac = solve_allen_cahn()
print("\nAllen-Cahn training complete.")

# Evaluate solution on grid
x_plot = torch.linspace(-1, 1, 100).reshape(-1, 1)
t_plot = torch.linspace(0, 1, 100).reshape(-1, 1)
X, T = torch.meshgrid(x_plot.squeeze(), t_plot.squeeze(), indexing='ij')

inputs = torch.stack([X.flatten(), T.flatten()], dim=1)
with torch.no_grad():
    U = model_ac(inputs[:, :1], inputs[:, 1:]).reshape(X.shape)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Solution contour
im = axes[0].contourf(X.numpy(), T.numpy(), U.detach().numpy(), levels=20)
axes[0].set_xlabel('x', fontsize=11)
axes[0].set_ylabel('t', fontsize=11)
axes[0].set_title('Allen-Cahn Solution u(x,t)', fontsize=12)
plt.colorbar(im, ax=axes[0])

# Loss curves
axes[1].semilogy(losses_ac, 'b-', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Total Loss', fontsize=11)
axes[1].set_title('Training Convergence', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Visualization of Training Dynamics

In [ ]:
# Checkpoint model at different epochs and visualize convergence
checkpoints = {}
epochs_to_save = [0, 50, 100, 200, 500, 1000]

# (Re-run training with checkpoints or load pre-trained models)
# Then create multi-panel visualization showing progression

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, epoch in enumerate(epochs_to_save):
    if epoch in checkpoints:
        model_ckpt = checkpoints[epoch]
        with torch.no_grad():
            U_ckpt = model_ckpt(inputs[:, :1], inputs[:, 1:]).reshape(X.shape)
        im = axes[idx].contourf(X.numpy(), T.numpy(), U_ckpt.numpy(),
                               levels=20)
        axes[idx].set_title(f'Epoch {epoch}', fontsize=11)
        axes[idx].set_xlabel('x', fontsize=10)
        axes[idx].set_ylabel('t', fontsize=10)
        plt.colorbar(im, ax=axes[idx])

plt.suptitle('Training Progression: Allen-Cahn Solution', fontsize=14)
plt.tight_layout()
plt.show()

## Comparison with Classical Methods

### Benchmark: 2D Poisson Equation

#### Finite Difference Method (FDM)

In [ ]:
import numpy as np
from scipy.sparse import diags, identity, kron
from scipy.sparse.linalg import spsolve

def solve_poisson_fdm(grid_size=100):
    """Solve 2D Poisson with the standard five-point stencil."""
    h = 1.0 / (grid_size + 1)

    # 1D second-difference matrix; 2D Laplacian via Kronecker products.
    # This construction avoids spurious couplings across grid rows.
    T = diags([1, -2, 1], [-1, 0, 1], shape=(grid_size, grid_size)) / h**2
    I = identity(grid_size)
    A = kron(I, T) + kron(T, I)  # discrete Laplacian on interior nodes

    # RHS: Laplacian of sin(2*pi*x)*sin(2*pi*y)
    x = np.linspace(h, 1 - h, grid_size)
    y = np.linspace(h, 1 - h, grid_size)
    X, Y = np.meshgrid(x, y)
    f = -8 * np.pi**2 * np.sin(2*np.pi*X) * np.sin(2*np.pi*Y)

    # Solve A u = f
    u_flat = spsolve(A.tocsr(), f.ravel())
    u = u_flat.reshape((grid_size, grid_size))

    return X, Y, u

# Benchmark
import time
t0 = time.time()
X_fdm, Y_fdm, u_fdm = solve_poisson_fdm(grid_size=200)
t_fdm = time.time() - t0
error_fdm = np.sqrt(np.mean((u_fdm - np.sin(2*np.pi*X_fdm)*np.sin(2*np.pi*Y_fdm))**2))
print(f"FDM (200x200): Time={t_fdm:.4f}s, L2 error={error_fdm:.6e}")